In [1]:
import cv2  # 图片处理三方库，用于对图片进行前后处理
import numpy as np  # 用于对多维数组进行计算

from mindx.sdk import Tensor  # mxVision 中的 Tensor 数据结构
from mindx.sdk import base  # mxVision 推理接口

# 可视化相关接口
import IPython.display
import ipywidgets as widgets

Begin to initialize Log.
The output directory of logs file exist.
Save logs information to specified directory.


I20260512 19:11:27.453210 63864 FileUtils.cpp:331] The input file is empty
I20260512 19:11:27.453298 63864 FileUtils.cpp:475] Check Other group permission: Current permission is 4, but required no greater than 0.


In [2]:
def preprocess(img_bgr):
    # 数据前处理
    img = cv2.resize(img_bgr, (256, 256)) # 对图像进行缩放
    img = img.astype(np.float32)
    mean = np.array([123.675, 116.28, 103.53], dtype=np.float32)     
    std = np.array([58.395, 57.12, 57.375], dtype=np.float32)
    img = (img - mean) / std
    img = img[:,:,::-1].transpose((2, 0, 1)) # BGR to RGB, HWC to CHW，将形状转换为 channel first 
    img = np.expand_dims(img, 0) # 得到(1, 3, 256, 256)，即扩展第一维为 batchsize
    img = np.ascontiguousarray(img, dtype=np.float32) # 转换为内存连续存储的数组
    img = Tensor(img) # 将numpy转为转为Tensor类

    return img

In [3]:
def infer(model, input_tensor):
    output_feat = model.infer([input_tensor])[0]

    return output_feat

In [4]:
def postprocess(img_bgr, output_feat):
    # 后处理
    img_width = img_bgr.shape[1]             
    img_height = img_bgr.shape[0]
    
    output_feat.to_host()  # 将Tensor数据转移到内存 
    output_feat_array = np.array(output_feat)[0]  # 将数据转为 numpy array 类型
    heatmaps_reshaped = output_feat_array.reshape((21, -1))
    idx = np.argmax(heatmaps_reshaped, 1).reshape((21, 1))
    maxvals = np.amax(heatmaps_reshaped, 1).reshape((21, 1))
    preds = np.tile(idx, (1, 2)).astype(np.float32)
    preds[:, 0] = preds[:, 0] % 64
    preds[:, 1] = preds[:, 1] // 64
    
    preds = np.where(np.tile(maxvals, (1, 2)) > 0.0, preds, -1)
    
    for k in range(21):
        heatmap = output_feat_array[k]
        
        px = int(preds[k][0])
        py = int(preds[k][1])
    
        if 1 < px < 64 - 1 and 1 < py < 64 - 1:
            diff = np.array([heatmap[py][px + 1] - heatmap[py][px - 1],
                             heatmap[py + 1][px] - heatmap[py - 1][px]])
            preds[k] += np.sign(diff) * .25
    
    all_preds = np.zeros((preds.shape[0], 3), dtype=np.float32)
    all_preds[:, 0:2] = preds
    all_preds[:, 2:3] = maxvals
    
    scale = np.array([img_width, img_height], dtype=np.float32)
    scale_x = scale[0] / 64
    scale_y = scale[1] / 64
    center = np.array([img_width / 2, img_height / 2], dtype=np.float32)
    all_preds[:, 0] = all_preds[:, 0] * scale_x + center[0] - scale[0] * 0.5
    all_preds[:, 1] = all_preds[:, 1] * scale_y + center[1] - scale[1] * 0.5
    
    pts_hand = {}
    for ptk, (xh, yh, score) in enumerate(all_preds):
        pts_hand[str(ptk)] = {"x": xh, "y": yh, "s": score}
        if score > 0.3:
            cv2.circle(img_bgr, (int(xh), int(yh)), 3, (255, 50, 60), -1)

    return img_bgr

In [5]:
def img2bytes(image):
    """将图片转换为字节码"""
    return bytes(cv2.imencode('.jpg', image)[1])

In [ ]:
# 初始化资源和变量
base.mx_init()  # 初始化 mxVision 资源
DEVICE_ID = 0  # 设备id

model_path = 'landmark.om'  # 模型路径
model = base.model(modelPath=model_path, deviceId=DEVICE_ID)  # 初始化 base.model 类

# 初始化摄像头
cap = cv2.VideoCapture(0)
# 初始化可视化对象
image_widget = widgets.Image(format='jpeg', width=1024, height=768)
display(image_widget)

try:
    while(cap.isOpened()):  # 在摄像头打开的情况下循环执行
        ret, frame = cap.read()  # 此处 frame 为 bgr 格式图片

        # ========================================
        # 这里放入模型前处理、推理、后处理相关代码，根据实际情况修改
        img = preprocess(frame)  # 前处理
        infer_res = infer(model, img)  # 推理
        img_res = postprocess(frame, infer_res)  # 后处理
        # ========================================

        # 将结果显示到notebook
        image_widget.value = img2bytes(img_res)
        
except KeyboardInterrupt:
    cap.release()  # 释放摄像头资源
finally:
    cap.release()

Image(value=b'', format='jpeg', height='768', width='1024')